In [28]:
import pandas as pd
import sqlite3
from io import StringIO


In [29]:

# =====================================================
# 1. FILE PATHS
# =====================================================
men_file = "./data/Nova Scotia Men population Age & Gender(2012-2025).csv"
women_file = "./data/Nova Scotia Women Population Age & Gender(2012 - 2025).csv"

output_csv = "./clean/clean_population.csv"
sqlite_db = "./database/population.db"

# =====================================================
# 2. SAFE STATSCAN TABLE READER
# =====================================================
def read_statscan_table(file_path):
    """
    Extracts the real StatsCan data table from a CSV file
    while ignoring metadata and malformed footnotes.
    """
    with open(file_path, "r", encoding="utf-8-sig", errors="ignore") as f:
        lines = f.readlines()

    start_idx = None
    end_idx = None

    for i, line in enumerate(lines):
        if start_idx is None and "Age group" in line:
            start_idx = i
        elif start_idx is not None and (
            "Median age" in line or "Footnotes" in line
        ):
            end_idx = i
            break

    if start_idx is None:
        raise ValueError("Could not locate StatsCan table header")

    if end_idx is None:
        end_idx = len(lines)

    table_text = "".join(lines[start_idx:end_idx])

    return pd.read_csv(StringIO(table_text))


# =====================================================
# 3. FILE CLEANING FUNCTION
# =====================================================
def process_file(file_path, gender_label):

    df = read_statscan_table(file_path)

    # Rename first column
    df.rename(columns={df.columns[0]: "Age_Group"}, inplace=True)

    # Remove non-data rows
    drop_rows = [
        "Persons", "All ages", "Median age", "Age group 3 6"
    ]
    df = df[~df["Age_Group"].isin(drop_rows)]
    df = df[df["Age_Group"].notna()]

    # Convert wide → long
    df_long = df.melt(
        id_vars="Age_Group",
        var_name="Year",
        value_name="Population"
    )

    # Clean numeric fields
    df_long["Population"] = (
        df_long["Population"]
        .astype(str)
        .str.replace(",", "", regex=False)
    )
    df_long["Population"] = pd.to_numeric(df_long["Population"], errors="coerce")
    df_long["Year"] = pd.to_numeric(df_long["Year"], errors="coerce")

    df_long = df_long.dropna(subset=["Year", "Population"])

    # Add gender
    df_long["Gender"] = gender_label

    # Final column order (EXACT MATCH)
    return df_long[["Year", "Age_Group", "Gender", "Population"]]


# =====================================================
# 4. PROCESS BOTH FILES
# =====================================================
men_df = process_file(men_file, "Male")
women_df = process_file(women_file, "Female")

final_df = pd.concat([men_df, women_df], ignore_index=True)

# =====================================================
# 5. SORT FOR READABILITY
# =====================================================
final_df = final_df.sort_values(
    by=["Age_Group", "Gender", "Year"]
)

# =====================================================
# 6. SAVE CLEAN CSV
# =====================================================
final_df.to_csv(output_csv, index=False)
print("✅ Clean CSV saved:", output_csv)

# =====================================================
# 7. SAVE TO SQLITE (SAFE, NO TABLE ERRORS)
# =====================================================
conn = sqlite3.connect(sqlite_db)
cursor = conn.cursor()

# Explicit drop avoids SQLite locking issues
cursor.execute("DROP TABLE IF EXISTS population;")
conn.commit()

final_df.to_sql(
    "population",
    conn,
    index=False
)

conn.close()
print("✅ Data loaded cleanly into SQLite:", sqlite_db)

✅ Clean CSV saved: ./clean/clean_population.csv
✅ Data loaded cleanly into SQLite: ./database/population.db
